# Building a Semantic Retrieval System for Book Discovery Using TF-IDF, NMF, and Cosine Similarity

**COSCI210 - Data Mining and Wrangling 1**

**Jose Miguel Bautista, PhD**
  
*Lab Report 2/Final Project*

**<u>MSDS 2026 Learning Team 8, Term 2</u>**

- Francis Erdey M. Capati
   
- Kevin Ansel S. Dy

- Jan Paolo V. Moreno

- Mia Cielo G. Oliveros

## Abstract
This paper presents an end-to-end semantic search pipeline designed to assist bookstore retailers in identifying fiction titles relevant to a customer?s requested themes or topics. Using Penguin Random House?s API as a metadata source, we construct a corpus of fiction titles and apply a multi-stage natural language processing (NLP) workflow. The flap-copy descriptions undergo TF-IDF vectorization, followed by Non-Negative Matrix Factorization (NMF) for topic extraction. A cosine similarity ranking layer enables semantic retrieval, allowing users to submit free-form text queries and receive the most relevant book titles. The resulting system enables accurate, interpretable topic-aligned book recommendations suitable for curation.

## Introduction and Problem Definition

A bookstore customer requests recommendations for titles aligned with specific themes or topics. The objective is to build a semantic retrieval system that returns relevant fiction books from the retailer?s inventory. The challenge is to interpret free-form text queries and identify the most semantically aligned titles using book metadata.

## Data Acquisition from Penguin Random House API

The corpus is constructed from Penguin Random House?s metadata service, accessed via a secure API key.

2.1. Collecting Fiction Category Identifiers

Fiction BISAC categories are retrieved from PRH endpoints and stored locally in a JSON file for reference.

These category IDs form the basis for selecting relevant books to include in the corpus.

2.2. Batch Retrieval of Title Metadata
A custom function accepts:

A list of category IDs

A record count per category

A set of API parameters

The function returns a list of data frames, each containing:

|ISBN|
|--|
|Title|
|Author|
|Flap-copy description|

These fields form the raw text corpus for downstream NLP processing.

In [ ]:
import requests
import time
from datetime import datetime, timedelta
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy import sparse
import json
import re
from sklearn.decomposition import NMF
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
API_KEY = "fh5hj47dynk4nvx4s9ewufj4"
BASE = "https://api.penguinrandomhouse.com/resources/v2/title/domains/PRH.US/titles/views/istca"
CAT = "https://api.penguinrandomhouse.com/resources/v2/domains/PRH.US/categories/"
TITLE_CAT = "https://api.penguinrandomhouse.com/resources/v2/title/domains/PRH.US/titles/"
session = requests.Session()

In [ ]:
BISAC = pd.read_json("bisac_prefixes.json", typ="series").reset_index()
BISAC.columns = ["prefix", "category"]
BISAC

In [ ]:
with open('fiction_BISAC.json', 'r') as f:
    fiction_bisac_codes = json.load(f)
fiction_bisac_codes = fiction_bisac_codes.get('data')
fiction_bisac_codes
f_cat_map = {
    c["catId"]: c["menuText"]
    # c["catId"]: {
    #     "BISAC": c["catUri"],
    #     "Description": c["menuText"]
        for c in fiction_bisac_codes["categories"]
}
f_cat_map

In [ ]:
len(f_cat_map)

In [ ]:
f_cat_id = {}
for key, value in f_cat_map.items():
    if value not in f_cat_id.values():
        f_cat_id[key] = value

f_cat_id

In [ ]:
f_cat_id[3000001771]

In [ ]:
def build_corpus(f_cat_id, rows=20):
    API_KEY = "fh5hj47dynk4nvx4s9ewufj4"
    BASE = "https://api.penguinrandomhouse.com/resources/v2/title/domains/PRH.US/titles/views/istca"
    CAT = "https://api.penguinrandomhouse.com/resources/v2/domains/PRH.US/categories/"
    session = requests.Session()

    results = []
    for catId in f_cat_id:

        params = {
            "formatFamily": "Paperback",
            "catId": catId,
            "showFlapCopy": "true",
            "showPublishedBooks": "true",
            "start": 0,
            "rows": rows,
            "api_key": API_KEY,
        }

        try:
            r = session.get(BASE, params=params, timeout=20)
            r.raise_for_status()

        except requests.exceptions.Timeout:
            #print(f"\n[TIMEOUT] Category {catId} timed out. Skipping.")
            continue

        except requests.exceptions.ConnectionError:
            #print(f"\n[CONNECTION ERROR] Failed to connect for category {catId}. Skipping.")
            continue

        except requests.exceptions.HTTPError as http_err:
            #print(f"\n[HTTP ERROR] Category {catId}: {http_err}. Skipping.")
            continue

        except Exception as e:
            #print(f"\n[UNKNOWN ERROR] Category {catId}: {e}. Skipping.")
            continue

        #print("\n---")
        #print("CATID:", catId)
        #print("STATUS:", r.status_code)
        #print("URL:", r.url)
        #print("RAW:", r.text[:200])

        r.raise_for_status()
        r_dict = r.json()
        for i in r_dict['data']:
            i['catId'] = catId
        results.append(r_dict)

    
    # Build the dataframe
    corpus = pd.DataFrame(results[0]['data'])
    #catid_corpus_dict = {}

    for catid in results:
        df = pd.DataFrame(catid['data'])
        try:
            corpus = pd.concat([corpus, df], ignore_index=True)
            #catid_corpus_dict[catid['params']['catId']] = len(catid['data'])
        except NameError:
            corpus = df

    corpus = corpus.drop(columns=['isbnHyphenated', 'workId', 'coverUrl',
        'format', 'subformat', 'binding', 'editionTarget', 'trim', 'edition',
        'onSaleDate', 'exportOnSaleDate', 'price', 'exportPrice',
        'globalDivision', 'publishingDivision', 'imprint', 'publishingStatus',
        'series', 'language', 'seq', 'titleBlock', 'authors'])

    return corpus

In [ ]:
corpus = build_corpus(f_cat_id, rows=100)
# for catid in fetch:
#     df = pd.DataFrame(catid['data'])
#     try:
#         corpus = pd.concat([corpus, df], ignore_index=True)
#     except NameError:
#         corpus = df

# corpus = corpus.drop(columns=['isbnHyphenated', 'workId', 'coverUrl',
#        'format', 'subformat', 'binding', 'editionTarget', 'trim', 'edition',
#        'onSaleDate', 'exportOnSaleDate', 'price', 'exportPrice',
#        'globalDivision', 'publishingDivision', 'imprint', 'publishingStatus',
#        'series', 'language', 'seq', 'titleBlock', 'authors'])
# # corpus.set_index('isbn', inplace=True)
corpus

In [ ]:
corpus = pd.read_json('fiction_corpus.json')
corpus = corpus.drop(columns=['isbnHyphenated', 'workId', 'coverUrl',
        'format', 'subformat', 'binding', 'editionTarget', 'trim', 'edition',
        'onSaleDate', 'exportOnSaleDate', 'price', 'exportPrice',
        'globalDivision', 'publishingDivision', 'imprint', 'publishingStatus',
        'series', 'language', 'seq', 'titleBlock', 'authors'])
corpus

In [ ]:
corpus.loc[:, 'description'] = (
    corpus['description']
        .str.replace(r"<.*?>", "", regex=True)            
        .str.replace(r"&[A-Za-z0-9#]+;", "", regex=True)  
        .str.replace(r"\d+", "", regex=True)              
        .str.strip()
)
corpus = corpus.drop_duplicates(subset=['title'], keep='first')
corpus.reset_index(drop=True, inplace=True)
corpus

In [ ]:
corpus.shape

#### Step 1. Establish corpus

In [ ]:
# Focus on descriptions only
working_corpus = corpus['description']
working_corpus

## TF-IDF Vectorization of Flap-Copy Descriptions

TF-IDF is applied to the book descriptions to numerically encode text.

Term Frequency (TF): Measures how often a word appears within a document, reflecting document-specific importance.

Inverse Document Frequency (IDF): Penalizes common words across the corpus while boosting rare, topic-specific terms.

The resulting sparse TF-IDF matrix serves as the foundation for topic modeling.

#### Step 2. Perform TF-IDF on corpus

In [ ]:
# TF-IDF
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(working_corpus)

# Get the feature names (words in the vocabulary)
feature_names = vectorizer.get_feature_names_out()

# Convert the sparse matrix to a dense array for easier viewing (for small datasets)
dense_matrix = tfidf_matrix.toarray()

# Create a DataFrame for better readability
tf_idf_df = pd.DataFrame(dense_matrix, columns=feature_names)
tf_idf_df

## Topic Extraction via SVD

In [ ]:
def truncated_svd(X):
    """
    Accepts a design matrix and returns q, sigma, p and the normalized sum of squared distance from the origin

    Inputs
    ------
    design matrix

    Returns
    -------
    q, sigma, p, normalized sum of squared distance from the origin
    """
    import numpy as np
    U, s, Vh = np.linalg.svd(X)
    
    # Get q
    q = U

    # Get sigma
    sigma = np.matmul(np.matmul(U.T, X), Vh.T)

    # Get p
    p = Vh.T

    # Get normalized sum of sqaured distance from the origin
    sq_singular_values = s**2
    nssd = sq_singular_values / sq_singular_values.sum()
    return q, sigma, p, nssd

In [ ]:
q, sigma, p, nssd = truncated_svd(tf_idf_df)

In [ ]:
fig, ax = plt.subplots()
ax.plot(range(1, len(nssd)+1), nssd, 'o-', label='individual')
ax.plot(range(1, len(nssd)+1), nssd.cumsum(), 'o-', label='cumulative')
ax.legend()
ax.set_ylim(0, 1)
ax.set_xlabel('SV')
ax.set_ylabel('variance explained');

## Topic Extraction via Non-Negative Matrix Factorization

Non-Negative Matrix Factorization (NMF) decomposes the TF-IDF matrix into:

A document-topic matrix (each book?s topic weights)

A topic-term matrix (keywords characterizing each topic)

NMF provides:

Dimensionality reduction

Interpretability

Topic clusters reflecting underlying themes in fiction descriptions

Optimal topic count may be evaluated using reconstruction error or coherence metrics.

#### Step 3. Perform NMF

In [ ]:
n_components = [5, 10, 15, 20, 30, 40, 50, 100]
errors = []
for n_component in n_components:
    nmf_ = NMF(n_component, max_iter=10000)
    nmf_.fit(tf_idf_df)
    errors.append(nmf_.reconstruction_err_)
plt.plot(n_components, errors, '-o')
plt.xlabel(r'$n_\text{components}$')
plt.ylabel('reconstruction error');

In [ ]:
n_topics = corpus['catId'].nunique()
n_topics

In [ ]:
# Instantiate the NMF model & specify the number of topics
# Set random_state for reproducibility
nmf_model = NMF(n_components=n_topics, random_state=1).set_output(transform="pandas")

# Fit the NMF model to the TF-IDF matrix
# Note that the 'H' matrix (topic-term distribution) is in nmf_model.components_
# Note that the 'W' matrix (document-topic distribution) can be obtained with model.transform(tfidf_matrix)
nmf_model.fit(tf_idf_df)

# Function to print the top words for each topic (from search results)
def print_top_words(model, feature_names, n_top_words):
    for topic_idx, topic in enumerate(nmf_model.components_):
        print(f"Topic #{topic_idx}:")
        print(" ".join([feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]]))
    print()

In [ ]:
# Print the top words per topic from search results
# print_top_words(nmf_model, feature_names, topic_no) # uncomment when using topic_no 
print_top_words(nmf_model, feature_names, 10)

In [ ]:
# Get the document-topic distribution
document_topic = nmf_model.transform(tf_idf_df)
document_topic

## Semantic Ranking Using Cosine Similarity

Cosine similarity measures the angular distance between the query?s projected topic vector and each book?s topic vector from the NMF model

This produces relevance scores and enables ranking the entire catalog by semantic similarity to the user?s query.

A reusable query function is implemented:

>Input: A natural-language text query

>Processing: TF-IDF transform ? NMF projection ? cosine similarity scoring

>Output: Top-10 semantically relevant book titles with metadata

This module forms the end-user interface for book discovery.

#### Step 4. Create a query function

In [ ]:
# Get top words per topic from search results
n_top_words = 10
topic_dict = {}
for topic_idx, topic in enumerate(nmf_model.components_):
        topic_dict[topic_idx] = [feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]]
topic_dict

#### Cosine Similarity Query

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def nmf_query_search(query, tfidf, nmf, W, top_k=20):

    # 1. Transform query using the SAME TF-IDF model
    q_tfidf = tfidf.transform([query])

    # 2. Project query into SAME NMF topic space
    q_vec = nmf.transform(q_tfidf)

    # 3. Cosine similarity
    sims = cosine_similarity(q_vec, W).ravel()

    # 4. Sort
    top_idx = sims.argsort()[::-1][:top_k]

    return top_idx, sims[top_idx]


In [ ]:
query = 'Romantic Comedy'

In [ ]:
idx, scores =  nmf_query_search(query=query, tfidf=vectorizer, nmf=nmf_model, W=document_topic, top_k=20)
# idx, scores
q_result = corpus.iloc[idx][['isbn', 'title', 'description']]
q_result

In [ ]:
def get_title_categories(isbn):
    params = {
        "api_key": API_KEY,
        "catSetId": "BI",
    }
    cat_url = f"{TITLE_CAT}{isbn}/categories"
    r = session.get(cat_url, params=params, timeout=20)
    print(r)
    print("STATUS:", r.status_code)
    print("URL:", r.url)
    print("RAW:", r.text[:250])

    result = dict(r.json())['data']['categories']
    out = []
    for entry in result:
        out.append(entry['catId'])

    r.raise_for_status()
    return out

In [ ]:
isbn = q_result.iloc[0]['isbn']
isbn

In [ ]:
get_title_categories(isbn)

In [ ]:
def build_catid_confusion_matrix(q_result, target_catid):
    """
    q_result: DataFrame containing ISBNs in column 'isbn'
    target_catid: int, the catId to evaluate

    Returns:
        confusion matrix (DataFrame with rows=Actual, cols=Predicted)
    """

    isbns = q_result["isbn"].tolist()

    rows = []

    for isbn in isbns:
        # Now returns a LIST of catIds
        cat_ids = get_title_categories(isbn)

        # Check if the target category is present
        has_cat = int(target_catid in cat_ids)

        rows.append({"isbn": isbn, "has_cat": has_cat})

    df = pd.DataFrame(rows)

    # All items in q_result are predicted relevant (1)
    df["predicted"] = 1

    # Build confusion matrix
    confusion = pd.crosstab(
        df["has_cat"],
        df["predicted"],
        rownames=["Actual"],
        colnames=["Predicted"]
    )

    return df, confusion

In [ ]:
build_catid_confusion_matrix(q_result, 3000001771)